In [16]:
import json
import os
import yaml
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional, Tuple, Union

import draccus
import torch
import torch.distributed as dist
from torch.utils.data import DataLoader, Dataset, DistributedSampler
from tqdm import tqdm
from transformers.modeling_outputs import CausalLMOutputWithPast
from transformers.models.auto import AutoConfig
from PIL import Image
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP

In [2]:
from prismatic.conf import DatasetConfig, DatasetRegistry, ModelConfig, ModelRegistry
from prismatic.models import get_llm_backbone_and_tokenizer, get_vision_backbone_and_transform, get_vlm
from prismatic.overwatch import initialize_overwatch
from prismatic.preprocessing import get_dataset_and_collator
from prismatic.training import Metrics, get_train_strategy
from prismatic.util import set_global_seed

# Load a checkpoint

In [3]:
checkpoint_path = "../runs/dino+siglip-llama-42/checkpoints/latest-checkpoint.pt"

In [4]:
checkpoint = torch.load(checkpoint_path, map_location="cuda" if torch.cuda.is_available() else "cpu")

In [5]:
def check_checkpoint(checkpoint_path: Path):
    print(f"[DEBUG] From {checkpoint_path}")
    print(f"[DEBUG] Checkpoint global_step = {checkpoint.get('global_step', 'MISSING')}")
    print(f"[DEBUG] Checkpoint epoch = {checkpoint.get('epoch', 'MISSING')}")
    print(f"[DEBUG] Checkpoint samples_seen = {checkpoint.get('samples_seen', 'MISSING')}")
    print(f"[DEBUG] Everything else...\n{checkpoint.keys()}")

In [6]:
check_checkpoint(checkpoint_path=checkpoint_path)

[DEBUG] From ../runs/dino+siglip-llama-42/checkpoints/latest-checkpoint.pt
[DEBUG] Checkpoint global_step = 2500
[DEBUG] Checkpoint epoch = 0
[DEBUG] Checkpoint samples_seen = 160000
[DEBUG] Everything else...
dict_keys(['model', 'optimizer', 'lr_scheduler', 'epoch', 'global_step', 'samples_seen', 'rng_state', 'train_loss'])


# Debug the Optimizer

In [13]:
len(checkpoint['optimizer']['state'])

1

In [18]:
checkpoint['optimizer']['state']

{0: {'step': tensor(2500., device='cuda:0'),
  'exp_avg': tensor([-1.5963e-07, -1.9706e-06, -8.6609e-07,  ..., -6.5393e-07,
          -3.1674e-07,  3.5618e-07], device='cuda:0'),
  'exp_avg_sq': tensor([9.5170e-11, 1.9516e-10, 1.2621e-10,  ..., 2.6558e-10, 9.5376e-10,
          4.0572e-11], device='cuda:0')}}

In [17]:
checkpoint['optimizer']['state'][0].keys()

dict_keys(['step', 'exp_avg', 'exp_avg_sq'])

In [15]:
checkpoint['optimizer']['state'][0]['exp_avg'].shape

torch.Size([17846400])